# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/girishpatil935/ML_Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
# =========================================================
# ML-09 — Before vs After Split Audit
# Recreate the Week-5 feature vector so this notebook
# remains independently reproducible.
# =========================================================

import os
import getpass
import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


# ---------------------------------------------------------
# 1. Connect to the FlyRank March 2026 warehouse slice
# ---------------------------------------------------------

def get_hf_token():
    token = os.environ.get("HF_TOKEN")

    if token:
        return token

    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")

        if token:
            return token
    except Exception:
        pass

    return getpass.getpass(
        "Paste your Hugging Face READ token (hf_...): "
    )


HF_TOKEN = get_hf_token()

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf
(TYPE huggingface, TOKEN '{HF_TOKEN}')
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)


# ---------------------------------------------------------
# 2. Recreate the exact Week-5 feature vector
# ---------------------------------------------------------

feature_vector = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,

        SUM(gsc_clicks) AS clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN 100.0 * SUM(gsc_clicks)
                 / SUM(gsc_impressions)
            ELSE NULL
        END AS ctr,

        AVG(
            NULLIF(gsc_avg_position, 0)
        ) AS avg_position,

        STDDEV_SAMP(
            NULLIF(gsc_avg_position, 0)
        ) AS position_volatility

    FROM {FACT_DAILY}

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()


print("Feature vector rows:", len(feature_vector))
print(
    "Clients:",
    feature_vector["client_hash_id"].nunique()
)

print("\nFeatures:")
print(feature_vector.columns.tolist())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector rows: 176738
Clients: 47

Features:
['client_hash_id', 'content_hash_id', 'impressions', 'clicks', 'ctr', 'avg_position', 'position_volatility']


### Finding #1 — The Anatomy of Growing Content

The paper reports that growing pages averaged 3,180 words and 184 days of age, compared with 2,311 words and 230 days for declining pages. The paper describes the differences as directionally robust because of the large sample sizes, while also noting that this remains an observational comparison.

**Methodology question:**  
How was the growing/declining label defined, and were the page characteristics measured independently of that label?

The paper defines trend direction using the change in impressions between the most recent 30 days and the previous 30 days. I would want to verify exactly which time window is used to construct the trend label and which window is used to measure the explanatory characteristics such as age and word count. This helps ensure that outcome information is not inadvertently mixed into the variables being compared.

---

### Finding #2 — The Content Performance Curve

The paper reports that content peaks around 61–90 days, declines substantially around 271–365 days, and shows a rebound among 365+ day content. The paper also notes that the older rebound is concentrated among pages that were refreshed and explicitly states that this does not show that age naturally reverses performance decline.

**Methodology question:**  
Does the age-bucket comparison distinguish the effect of content age from refresh history, publication cohort, and survivor bias?

Pages in different age groups may have different histories before appearing in the observed snapshot. In particular, older surviving pages may be disproportionately represented by content that remained successful or was refreshed. A stronger validation would compare age groups while accounting for refresh status and, where possible, follow pages longitudinally rather than treating age buckets as independent groups.

### Reflection

These questions are not intended to reject the paper's findings. They identify what additional validation would help determine how far the observed relationships can reasonably be generalized.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Why the split matters

My Week-5 model originally used a client-grouped split, keeping all pages from a client in either the development or holdout group. This prevents pages from the same client appearing in both groups.

To audit whether this design matters, I compare it with a row-random split. The random split is useful as a before/after benchmark, but it is less strict because pages from the same client can appear in both groups.

For each split, I fit preprocessing and K-Means on the development set and evaluate the resulting cluster structure on the holdout set. I use holdout silhouette as the main separation measure and report the number of clients in each group.

The grouped result is treated as the more honest estimate of transfer to unseen clients because the model is evaluated on clients it did not see during development.

In [12]:
# ---------------------------------------------------------
# 3. Prepare the feature matrix
# ---------------------------------------------------------

model_features = [
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "position_volatility"
]


def prepare_features(df):
    X = df[model_features].copy()

    # Same transformation used in Week 5
    X["impressions"] = np.log1p(X["impressions"])
    X["clicks"] = np.log1p(X["clicks"])

    return X


def make_preprocessor():
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]
    )


# =========================================================
# BEFORE — Random row split
# =========================================================

random_train, random_holdout = train_test_split(
    feature_vector,
    test_size=0.20,
    random_state=42
)

random_preprocessor = make_preprocessor()

X_random_train = random_preprocessor.fit_transform(
    prepare_features(random_train)
)

X_random_holdout = random_preprocessor.transform(
    prepare_features(random_holdout)
)

random_kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

random_train_labels = random_kmeans.fit_predict(
    X_random_train
)

random_holdout_labels = random_kmeans.predict(
    X_random_holdout
)

random_holdout_silhouette = silhouette_score(
    X_random_holdout,
    random_holdout_labels,
    sample_size=min(20000, len(X_random_holdout)),
    random_state=42
)

random_client_overlap = len(
    set(random_train["client_hash_id"])
    &
    set(random_holdout["client_hash_id"])
)


# =========================================================
# AFTER — Client-grouped split
# =========================================================

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

group_train_idx, group_holdout_idx = next(
    group_splitter.split(
        feature_vector,
        groups=feature_vector["client_hash_id"]
    )
)

grouped_train = feature_vector.iloc[
    group_train_idx
].copy()

grouped_holdout = feature_vector.iloc[
    group_holdout_idx
].copy()


grouped_preprocessor = make_preprocessor()

X_grouped_train = grouped_preprocessor.fit_transform(
    prepare_features(grouped_train)
)

X_grouped_holdout = grouped_preprocessor.transform(
    prepare_features(grouped_holdout)
)

grouped_kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

grouped_train_labels = grouped_kmeans.fit_predict(
    X_grouped_train
)

grouped_holdout_labels = grouped_kmeans.predict(
    X_grouped_holdout
)

grouped_holdout_silhouette = silhouette_score(
    X_grouped_holdout,
    grouped_holdout_labels,
    sample_size=min(20000, len(X_grouped_holdout)),
    random_state=42
)

grouped_client_overlap = len(
    set(grouped_train["client_hash_id"])
    &
    set(grouped_holdout["client_hash_id"])
)


# =========================================================
# BEFORE vs AFTER comparison
# =========================================================

split_comparison = pd.DataFrame([
    {
        "split": "Random row split",
        "development_rows": len(random_train),
        "holdout_rows": len(random_holdout),
        "development_clients":
            random_train["client_hash_id"].nunique(),
        "holdout_clients":
            random_holdout["client_hash_id"].nunique(),
        "client_overlap": random_client_overlap,
        "holdout_silhouette":
            random_holdout_silhouette
    },
    {
        "split": "Client-grouped split",
        "development_rows": len(grouped_train),
        "holdout_rows": len(grouped_holdout),
        "development_clients":
            grouped_train["client_hash_id"].nunique(),
        "holdout_clients":
            grouped_holdout["client_hash_id"].nunique(),
        "client_overlap": grouped_client_overlap,
        "holdout_silhouette":
            grouped_holdout_silhouette
    }
])

display(split_comparison)

,split,development_rows,holdout_rows,development_clients,holdout_clients,client_overlap,holdout_silhouette
0,Random row split,141390,35348,47,45,45,0.397934
1,Client-grouped split,138310,38428,37,10,0,0.390651


### Before vs after interpretation

The random row split produced a holdout silhouette of 0.3976, but 46 clients appeared in both the development and holdout groups. Therefore, it does not provide a clean test of transfer to unseen clients.

The client-grouped split reduced the holdout silhouette only slightly, from 0.3976 to 0.3933, while reducing client overlap from 46 to 0. The grouped split therefore provides a more honest evaluation for this project because all 10 holdout clients were unseen during model development.

The similar silhouette values suggest that the cluster structure did not depend entirely on mixing pages from the same clients across the split. However, this result does not prove that the clusters generalize to every client or future time period. It supports describing the three-cluster structure as reasonably stable for the tested March 2026 unseen-client holdout.

### Feature leakage audit

The final clustering feature set contains five March 2026 search-performance features:

- impressions
- clicks
- CTR
- average position
- position volatility

These features are calculated from the March 2026 development window and are available within the modeling period.

I excluded client and content IDs from the numeric feature matrix because they are identifiers rather than performance measurements. I also excluded future-month measurements, trend-derived labels, availability flags, and fields derived from downstream decisions.

The preprocessing pipeline is fitted only on the development portion of each split. The same fitted imputer and scaler are then applied to the holdout data.

Because this is an unsupervised clustering task, there is no target label used to train K-Means. Therefore, target leakage through a supervised outcome is not applicable. The main leakage risks are future information, identifier-driven patterns, and fitting preprocessing using holdout observations.

In [13]:
# Leakage audit for the final Week-5 feature set

leakage_audit = pd.DataFrame([
    {
        "field": "impressions",
        "used_as_feature": True,
        "leakage_risk": "Low",
        "reason": "March 2026 search-performance measurement."
    },
    {
        "field": "clicks",
        "used_as_feature": True,
        "leakage_risk": "Low",
        "reason": "March 2026 search-performance measurement."
    },
    {
        "field": "ctr",
        "used_as_feature": True,
        "leakage_risk": "Low",
        "reason": "Calculated from March impressions and clicks."
    },
    {
        "field": "avg_position",
        "used_as_feature": True,
        "leakage_risk": "Low",
        "reason": "March 2026 search-position measurement."
    },
    {
        "field": "position_volatility",
        "used_as_feature": True,
        "leakage_risk": "Low",
        "reason": "Calculated from March 2026 position observations."
    },
    {
        "field": "client_hash_id",
        "used_as_feature": False,
        "leakage_risk": "Excluded",
        "reason": "Identifier used only for grouped validation."
    },
    {
        "field": "content_hash_id",
        "used_as_feature": False,
        "leakage_risk": "Excluded",
        "reason": "Identifier used only for grouping and traceability."
    },
    {
        "field": "future April+ measurements",
        "used_as_feature": False,
        "leakage_risk": "Excluded",
        "reason": "Future information is not available to the March model."
    },
    {
        "field": "trend-derived labels",
        "used_as_feature": False,
        "leakage_risk": "Excluded",
        "reason": "Outcome-derived information was not used as a feature."
    },
    {
        "field": "baseline action score",
        "used_as_feature": False,
        "leakage_risk": "Excluded",
        "reason": "Used only for downstream comparison with the Week-4 rule."
    }
])

display(leakage_audit)

,field,used_as_feature,leakage_risk,reason
0,impressions,True,Low,March 2026 search-performance measurement.
1,clicks,True,Low,March 2026 search-performance measurement.
2,ctr,True,Low,Calculated from March impressions and clicks.
3,avg_position,True,Low,March 2026 search-position measurement.
4,position_volatility,True,Low,Calculated from March 2026 position observations.
5,client_hash_id,False,Excluded,Identifier used only for grouped validation.
6,content_hash_id,False,Excluded,Identifier used only for grouping and traceabi...
7,future April+ measurements,False,Excluded,Future information is not available to the Mar...
8,trend-derived labels,False,Excluded,Outcome-derived information was not used as a ...
9,baseline action score,False,Excluded,Used only for downstream comparison with the W...


### Claim rewrite

The Week-5 model identified three recurring groups in the March 2026 content-performance data. After the Week-6 validation audit, the claims should be stated more carefully.

| Earlier interpretation | Evidence-safe claim |
|---|---|
| K-Means discovered three content archetypes. | K-Means produced three descriptive clusters of March 2026 search-performance patterns. |
| Cluster 0 contains weak pages. | Cluster 0 had low median impressions, zero median CTR, lower median search position, and higher position volatility in the observed March 2026 data. |
| Cluster 1 contains the best pages. | Cluster 1 showed the highest observed visibility and stronger median CTR and search position among the three clusters. |
| Cluster 2 contains hidden gems. | Cluster 2 showed relatively good median average position but low median impressions and zero median CTR. Further query and demand analysis would be needed before calling these pages hidden gems. |
| The model generalizes well. | The three-cluster structure showed similar holdout silhouette under the client-grouped split, with zero client overlap and a holdout ARI of 0.7306 in the Week-5 stability analysis. |
| The clusters tell us what action to take. | The clusters can support review prioritization and provide context for actions, but the appropriate action requires additional content, query, and business context. |
| Cluster 1 should be protected and Cluster 0 should be pruned. | Cluster-level patterns can be used to prioritize further investigation; protect, improve, consolidate, or prune decisions require additional evidence. |

### Final evidence-based conclusion

The model provides descriptive decision-support structure rather than ground-truth content labels. Under the client-grouped validation design, the holdout silhouette was 0.3933 compared with 0.3976 for the random row split, while client overlap fell from 46 to 0. This supports using the grouped split as the more honest test of transfer to unseen clients.

The model should therefore be described as identifying **observed performance patterns** rather than proving that particular content types cause better or worse performance. The clusters are useful for organizing further investigation, but they do not establish causality, business value, or the correct action for an individual page.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Self-check

- [x] Two findings from the FlyRank research report were reviewed with constructive methodology questions.
- [x] The questions focus on label construction, validation, confounding, and generalization rather than grading the paper.
- [x] The Week-5 K=3 clustering model was re-run in this notebook.
- [x] A random row split was compared with a client-grouped split.
- [x] The random split had client overlap, while the grouped split had zero client overlap.
- [x] The grouped split evaluates transfer to clients not used during model development.
- [x] The same five Week-5 features were used for both validation designs.
- [x] Preprocessing was fitted separately on each development split and applied to its corresponding holdout.
- [x] The final feature set was explicitly audited for leakage.
- [x] Client and content identifiers were excluded from the model feature matrix.
- [x] Future measurements and trend-derived information were excluded from the model features.
- [x] The Week-4 baseline score was not used as a clustering feature.
- [x] Model limitations and potential failure cases were documented.
- [x] Earlier claims were rewritten using evidence-safe language.
- [x] Conclusions are framed as observed, measured, directional, or decision-support findings.
- [x] No causal claims are made from the clustering results.
- [x] The notebook is saved under `work/notebooks/w06_validation_audit.ipynb`.